# Modelo Computacional para Clientes Heterogêneos

In [1]:
from random import randint
from pprint import pprint

class Client:
    def __init__(
        self,
        Eio:float,          # initial energy
        Bi:int,             # num batches
        gamma_i:float,      # CPU effective capacitance
        ci:int,             # CPU cycles for processing 1 batch
        fi:int,             # CPU clock freq
        ui:float,           # upload time
        di:float,           # download time
        epochs:int,         # num epochs
    ) -> None:
        
        self.Eio = Eio
        self.ui = ui
        self.di = di

        self.epsilon_i = Bi*ci*gamma_i*fi*fi # energy per epoch
        self.tau_i = Bi*ci/fi # time per epoch (bacthes * cycles/batch / cycles/sec = sec)
       
        self.Ti = epochs*self.tau_i + self.ui + self.di 
        self.num_epochs = epochs
        self.energy_consumed_training = self.epsilon_i * self.num_epochs
        self.Ei = self.Eio - self.energy_consumed_training

    def print_report(self):
        print("New client created:")
        print(f"Energy per epoch = {self.epsilon_i} J")
        print(f"Time per epoch = {self.tau_i}s")
        print(f"Num epochs: {self.num_epochs}")
        print(f"Time: {self.Ti} s")
        print(f"Energy before: {self.Eio} J")
        print(f"Energy consumed by training: {self.energy_consumed_training} J")
        print(f"Energy after: {self.Ei} J")

## Função para gerar características dos dispositivos segundo distribuição
* Em experimentos científicos, pode-se usar outras distribuições que não a uniforme 
* Em implantações reais, essas características seriam coletadas dos próprios dispositivos

In [2]:
def generate_random_specs():
    battery_mAh = randint(1900, 2200)
    battery_soc = randint(10, 40)
    num_batches = randint(10, 20)
    cpu_GHz = randint(28, 32) / 10
    ci_est = randint(24, 36) * 1e10
    gamma_est = randint(50, 55) * 1e-38
    upload_Mbps = randint(40, 60)
    download_Mbps = upload_Mbps
    
    return {
        "battery_mAh": battery_mAh,
        "battery_soc": battery_soc,
        "num_batches": num_batches,
        "cpu_GHz": cpu_GHz,
        "ci_est": ci_est,
        "gamma_est": gamma_est,
        "upload_Mbps": upload_Mbps,
        "download_Mbps": download_Mbps
    }

## Função para calcular o consumo de energia e tempo de treinamento de clientes aleatórios
* A função gera as carcaterísticas dos clientes usando a função antrior
* Assume-se que o número de épocas pode ser diferente para cada cliente

### def gen_clients(num_clients, epochs_list):
    assert len(epochs_list) == num_clients, "The length of epochs_list must match num_clients"
    clients = []
    for i in range(num_clients):
        specs = generate_random_specs()

        print(f"\nClient {i} specs:")
        pprint(specs)

        Eio = specs['battery_mAh'] * 1e-3 * 3600 * 3.7
        client = Client(
            Eio=Eio,
            Bi=specs['num_batches'],
            gamma_i=specs['gamma_est'],
            ci=specs['ci_est'],
            fi=specs['cpu_GHz'] * 1e9,
            ui=10 / specs['upload_Mbps'], # tempo de upload em segundos
            di=10 / specs['download_Mbps'], # tempo de download em segundos
            epochs=epochs_list[i] # número de épocas (pode ser ajustado conforme necessário)
        )
        clients.append(client)
    return clients

## Gera 5 clientes (3 épocas locais em todos)

In [4]:
clients_list = gen_clients(5, [3,3,3,3,3])


Client 0 specs:
{'battery_mAh': 1961,
 'battery_soc': 15,
 'ci_est': 290000000000.0,
 'cpu_GHz': 3.0,
 'download_Mbps': 52,
 'gamma_est': 5.3e-37,
 'num_batches': 11,
 'upload_Mbps': 52}

Client 1 specs:
{'battery_mAh': 2146,
 'battery_soc': 15,
 'ci_est': 280000000000.0,
 'cpu_GHz': 3.0,
 'download_Mbps': 60,
 'gamma_est': 5.3999999999999995e-37,
 'num_batches': 15,
 'upload_Mbps': 60}

Client 2 specs:
{'battery_mAh': 2000,
 'battery_soc': 39,
 'ci_est': 320000000000.0,
 'cpu_GHz': 3.2,
 'download_Mbps': 52,
 'gamma_est': 5.1e-37,
 'num_batches': 19,
 'upload_Mbps': 52}

Client 3 specs:
{'battery_mAh': 2099,
 'battery_soc': 40,
 'ci_est': 360000000000.0,
 'cpu_GHz': 3.0,
 'download_Mbps': 59,
 'gamma_est': 5.5e-37,
 'num_batches': 12,
 'upload_Mbps': 59}

Client 4 specs:
{'battery_mAh': 1956,
 'battery_soc': 24,
 'ci_est': 320000000000.0,
 'cpu_GHz': 3.2,
 'download_Mbps': 46,
 'gamma_est': 5.3999999999999995e-37,
 'num_batches': 13,
 'upload_Mbps': 46}


## Imprime relatório final para cada cliente

In [5]:
for i,client in enumerate(clients_list):
    print(f"\nClient {i} Report:")
    client.print_report()


Client 0 Report:
New client created:
Energy per epoch = 1.5216299999999999e-05 J
Time per epoch = 1063.3333333333333s
Num epochs: 3
Time: 3190.3846153846152 s
Energy before: 26120.520000000004 J
Energy consumed by training: 4.5648899999999996e-05 J
Energy after: 26120.519954351104 J

Client 1 Report:
New client created:
Energy per epoch = 2.0411999999999997e-05 J
Time per epoch = 1400.0s
Num epochs: 3
Time: 4200.333333333334 s
Energy before: 28584.72 J
Energy consumed by training: 6.1236e-05 J
Energy after: 28584.719938764 J

Client 2 Report:
New client created:
Energy per epoch = 3.1752192e-05 J
Time per epoch = 1900.0s
Num epochs: 3
Time: 5700.384615384615 s
Energy before: 26640.0 J
Energy consumed by training: 9.5256576e-05 J
Energy after: 26639.999904743425 J

Client 3 Report:
New client created:
Energy per epoch = 2.1383999999999997e-05 J
Time per epoch = 1440.0s
Num epochs: 3
Time: 4320.338983050848 s
Energy before: 27958.680000000004 J
Energy consumed by training: 6.41519999999